# Process a mixed monocular + disparity goggles session

`teensyStimulusAngle(..., 'mode', 'mixed')` intermingles binocular disparity
trials and monocular (one-eye) trials in a single recording, and logs both into
one stimulus `.mat`. The rest of the pipeline expects one paradigm per session
directory.

This notebook splits that session into two derived session directories that look
exactly like separately-recorded sessions, then runs the **existing** process
steps on each. The outputs are the same `{session_name}_dataframe.pkl` files that
`process-disparity-session.ipynb` and `process-monocular-session.ipynb` produce,
and that `monocular-disparity-distance-comparison.ipynb` reads via
`CrossExperimentAnalysis.from_devices(R, M, D)`.

    260916_SAM046_goggles/      source, untouched
    260916_SAM046_disparity/    binocular trials -> DisparityDevice
    260916_SAM046_monocular/    monocular trials -> MonocularDevice

**The split is required, not cosmetic.** Monocular trials are logged with
`trial_disparity_deg = 0`, so pointing `DisparityDevice` at the mixed session
silently folds them into the D=0 condition -- for this session that is 45 trials
at D=0 of which only 15 are actually binocular. `MonocularDevice` has the mirror
problem: binocular trials carry `eye_order = 0`, which is not a valid eye code.

Both derived sessions share the source `suite2p` output through a directory
junction. The two paradigms are one recording and trials are selected by time
window, not by frame slice, so no activity data is duplicated or split -- and
`iscell.npy` curation stays shared between them.

Run this in its own kernel with **only** `disparity-analysis` on `sys.path`.

In [ ]:
import sys, os
import pandas as pd

base_dir = r"C:\Users\acostamendoza\projects\distance-project"
for p in (base_dir, base_dir + r"\disparity-analysis"):
    if p not in sys.path:
        sys.path.append(p)

from src.data.split_mixed_session import split_mixed_session
from src.data.DisparityDevice import DisparityDevice
from src.data.MonocularDevice import MonocularDevice
from src.analysis.DisparityAnalysis import DisparitySessionAnalyzer
from src.analysis.CellAnalysis import SessionAnalyzer

%load_ext autoreload
%autoreload 2

## 1. Split the mixed session

Writes `{date}_{mouse}_disparity` and `{date}_{mouse}_monocular` next to the
source. Re-running is safe: an existing stimulus `.mat` is left alone unless
`overwrite=True`, and the suite2p junction is only created once.

In [ ]:
mouse, date, experiment = "SAM046", "260916", "goggles"

base_data_dir = r"D:/data-distance-coding-project/processed-data"
session_dir = os.path.join(base_data_dir, f"{date}_{mouse}_{experiment}")

split_dirs = split_mixed_session(session_dir)
split_dirs

## 2. Disparity half

Identical to `process-disparity-session.ipynb`, just pointed at the derived
directory.

In [ ]:
D = DisparityDevice(data_dir=split_dirs["disparity"])
print("session:", D.session_name, "| disparities:", D.disparities, "| cells:", len(D.suite2p_id))
print("trials per disparity:", {d: len(D.get_disparity_repeats(d)) for d in D.disparities})

In [ ]:
# load cached stats if present, else compute once (slow) and save the pkl the
# comparison notebook reads
pkl_path = os.path.join(D.data_dir, f"{D.session_name}_dataframe.pkl")
if os.path.exists(pkl_path):
    disparity_df = pd.read_pickle(pkl_path)
    print(f"Loaded cached stats: {pkl_path}")
else:
    disparity_df = DisparitySessionAnalyzer(D).analyze_session_stats(n_iterations=200)
    DisparitySessionAnalyzer(D).save_dataframe(disparity_df)   # -> pkl_path
disparity_df.head()

## 3. Monocular half

Identical to `process-monocular-session.ipynb`, pointed at the other derived
directory.

In [ ]:
M = MonocularDevice(data_dir=split_dirs["monocular"])
print("session:", M.session_name, "| eyes:", M.present_eyes, "| cells:", len(M.suite2p_id))
print("trials per eye:", {e: len(M.get_eye_repeats(e)) for e in M.present_eyes})

In [ ]:
pkl_path = os.path.join(M.data_dir, f"{M.session_name}_dataframe.pkl")
if os.path.exists(pkl_path):
    monocular_df = pd.read_pickle(pkl_path)
    print(f"Loaded cached stats: {pkl_path}")
else:
    monocular_df = SessionAnalyzer(M).analyze_session_stats(n_iterations=200)
    SessionAnalyzer(M).save_dataframe(monocular_df)   # -> pkl_path
monocular_df.head()

## 4. Check the split conserved the session

Trial counts across the two halves must add back up to the mixed session, and
both halves must sit on the same cells -- they are the same recording.

In [ ]:
from utils.matlab_io import load_mat_struct
import numpy as np

stim_mat = [f for f in os.listdir(os.path.join(session_dir, "stimulus")) if f.endswith(".mat")][0]
mixed = load_mat_struct(os.path.join(session_dir, "stimulus", stim_mat))
n_mixed = len(np.asarray(mixed["trial_kind"]).ravel())

print(f"mixed session   : {n_mixed:4d} trials  (mode={mixed['mode']})")
print(f"  binocular     : {D.n_repeats:4d}  = {len(D.disparities)} disparities x "
      f"{len(D.get_disparity_repeats(D.disparities[0]))} repeats")
print(f"  monocular     : {M.n_repeats:4d}  = {len(M.present_eyes)} eyes x "
      f"{len(M.get_eye_repeats(M.present_eyes[0]))} repeats")
print(f"  total         : {D.n_repeats + M.n_repeats:4d}")
print(f"cells (shared)  : {len(D.suite2p_id):4d}")

assert D.n_repeats + M.n_repeats == n_mixed, "trials lost or duplicated in the split"
assert len(D.suite2p_id) == len(M.suite2p_id), "halves disagree on the cell set"
assert set(np.unique(M.eye_order)) <= {1, 2}, "binocular trials leaked into the monocular half"
print("\nOK - split conserves trials and cells")

## 5. Standard monocular report (optional)

Same as `process-monocular-session.ipynb`. Slow when `plot_cells=True`.

In [ ]:
from src.viz.SessionVisualizer import SessionVisualizer

sv = SessionVisualizer(M, stats_df=monocular_df, corr_threshold=0.7)
sets = sv.save_report(plot_cells=False)

reliable_cells = sorted(sets["either"])    # reliable in left OR right eye
print(f"{len(reliable_cells)} cells reliable in at least one eye")